# Chronos -- HYBRID training: EfficientNet + ConvLSTM + fusion (THE model)

Trains the thesis model **jointly, end to end, in one run**:

```
window of 16 contiguous frames (B, T, C, H, W)
  ├─ EfficientNet per frame ──► per-frame features
  │     ├─ spatial head  ──► spatial logit   (single-frame artifacts:
  │     │                                     textures, watermarks, seams)
  │     └─ ConvLSTM over time ─► temporal head ─► temporal logit
  │                                           (motion artifacts: flicker,
  │                                            identity drift, physics)
  └─ fusion head over [spatial + temporal features] ──► FUSED logit  <- verdict
```

The fused logit is the model's answer; the per-branch logits are auxiliary
training signals (`aux_loss_weight`) that force each branch to stay
independently discriminative -- and the final eval reports fused vs spatial
vs temporal AUC, which is the branch-contribution table for the thesis.

Consumes the SAME extraction output as before (`chronos_frames_*.tar` + index
CSVs; ALL attached tars are merged) -- the contiguous windows in it were built
exactly for this.

## Before you run
1. **Add Input -> Your Work** -> select the extraction notebook.
2. **Accelerator: GPU P100 (or T4)**. **Internet: ON** (timm, albumentations).

## Do this first: SMOKE TEST
Leave `SMOKE_TEST = True` for the first run -- small backbone, ~120-video
subsample, 3 total epochs: proves manifest -> windows -> hybrid -> train ->
eval end to end in ~15 min. Then flip to `SMOKE_TEST = False` and Commit.

In [ ]:
# ========================= DISCOVER MOUNTED INPUTS ==========================
!ls -la /kaggle/input/

In [ ]:
# ============================== LOCATE EXTRACTION OUTPUT ====================
# Two supported input forms:
#   A. TAR(s):  chronos_frames_*.tar  (extraction notebook output attached
#      directly; several partial tars merge fine)
#   B. PRE-EXTRACTED DATASET: a dataset holding frames/ + frames_index*.csv
#      loose -- this happens when the tar is re-uploaded as a Kaggle DATASET,
#      because Kaggle auto-extracts uploaded archives.
import glob
from pathlib import Path

tar_matches = sorted(set(glob.glob("/kaggle/input/**/chronos_frames_*.tar", recursive=True)))
TAR_PATHS = [Path(p) for p in tar_matches]

# form B: a frames_index*.csv WITH a sibling frames/ directory
_idx = glob.glob("/kaggle/input/**/frames_index*.csv", recursive=True)
EXTRACTED_ROOTS = sorted({str(Path(p).parent) for p in _idx
                          if (Path(p).parent / "frames").is_dir()})

if not TAR_PATHS and not EXTRACTED_ROOTS:
    raise FileNotFoundError(
        "No extraction data found under /kaggle/input/**. Attach EITHER the "
        "extraction notebook's output (chronos_frames_*.tar) OR a dataset "
        "containing frames/ + frames_index*.csv."
    )
if TAR_PATHS:
    print(f"Found {len(TAR_PATHS)} extraction tar(s):")
    for p in TAR_PATHS:
        print("  ", p)
else:
    print(f"No tar; using pre-extracted dataset(s): {EXTRACTED_ROOTS}")

In [ ]:
# ============================== CONFIG =======================================
SMOKE_TEST = True   # <<< EDIT ME: True = fast pipeline check (~10-15 min). False = full run.

SEED = 42
# DISK LAYOUT: frames untar to the SCRATCH disk (/kaggle/tmp -- big, NOT
# persisted). Everything under /kaggle/working becomes the notebook's output
# zip, so keeping frames there bloated the output to ~14GB of data the next
# stage never needs. Output now contains ONLY outputs/ (checkpoints+metrics).
DATA_ROOT = Path("/kaggle/tmp/data")
OUTPUTS_DIR = Path("/kaggle/working/outputs")

# CFG mirrors configs/v1_spatial.yaml so the ported functions below (which
# index cfg["train"]["lr_head"] etc.) work unchanged.
# NOTE: "extraction" and "inference" sections are nested exactly like
# configs/v1_spatial.yaml (the local pipeline's config) -- inference.py
# reads cfg["extraction"]["image_size"] and cfg["inference"][...], and this
# CFG gets embedded verbatim into the checkpoint as ckpt["config"], so
# matching the shape here is what makes a downloaded best.pt load correctly
# in inference.py / the local server later.
if SMOKE_TEST:
    CFG = {
        "seed": SEED,
        "paths": {"data_root": str(DATA_ROOT), "manifest": str(DATA_ROOT / "manifest.csv"),
                  "outputs_dir": str(OUTPUTS_DIR)},
        # trim_black_borders is stored in the checkpoint so inference.py applies
        # the SAME bar-trim the training frames got (preprocessing parity).
        "extraction": {"image_size": 224, "trim_black_borders": True},   # smaller/faster for the smoke pass; frames were saved at 380px, so this just downsamples on load
        # scene_regex groups a portrait crop ("<orig>_pcrop_<hash>") with its
        # landscape original into ONE split -- same content, splitting leaks.
        "splits": {"train": 0.70, "val": 0.15, "test": 0.15, "scene_regex": r"^(.+?)(?:_pcrop(?:_[0-9a-f]+)?)?$", "cap_per_class": 11500, "trim_from": "hybridframeextraction"},
        "model": {"arch": "hybrid", "backbone": "tf_efficientnet_b0_ns", "pretrained": True, "dropout": 0.3,
                  "convlstm": {"hidden_channels": 64, "kernel_size": 3}, "fusion_hidden": 64,
                  "motion_branch": True, "motion_dim": 64, "frequency_branch": False, "logit_fusion": True},  # smoke exercises motion
        # batch_videos = windows per step (each = window_len frames through the
        # backbone) -- THE memory knob for the hybrid.
        "train": {"batch_videos": 6, "num_workers": 2, "epochs_head": 1, "epochs_finetune": 2,
                  "lr_head": 1e-3, "lr_finetune_head": 1e-4, "lr_backbone": 1e-5,
                  "weight_decay": 1e-4, "unfreeze_blocks": 3, "early_stop_patience": 5,
                  "aux_loss_weight": 0.3, "motion_aux_weight": 0.6, "use_weighted_sampler": False, "amp": True},
        "augment": {"aspect_jitter": True, "portrait_squash": True, "horizontal_flip": True,
                    "lighting": True, "resolution": True, "compression": True, "blur": True, "noise": True, "overlays": True},
        "inference": {"frames_per_video": 32, "cam_threshold": 0.60, "max_boxes": 3,
                      "verdict_real_below": 35, "verdict_fake_above": 65},
        "smoke_videos_per_source": 60,
    }
else:
    CFG = {
        "seed": SEED,
        "paths": {"data_root": str(DATA_ROOT), "manifest": str(DATA_ROOT / "manifest.csv"),
                  "outputs_dir": str(OUTPUTS_DIR)},
        # trim_black_borders stored in the checkpoint -> inference.py bar-trims
        # identically (preprocessing parity).
        "extraction": {"image_size": 380, "trim_black_borders": True},   # matches the square size the frames were saved at
        # scene_regex groups a portrait crop ("<orig>_pcrop_<hash>") with its
        # landscape original into ONE split -- same content, splitting leaks.
        "splits": {"train": 0.70, "val": 0.15, "test": 0.15, "scene_regex": r"^(.+?)(?:_pcrop(?:_[0-9a-f]+)?)?$", "cap_per_class": 11500, "trim_from": "hybridframeextraction"},
        "model": {"arch": "hybrid", "backbone": "tf_efficientnet_b4_ns", "pretrained": True, "dropout": 0.4,
                  "convlstm": {"hidden_channels": 256, "kernel_size": 3}, "fusion_hidden": 256,
                  # 3rd branch = MOTION: temporal-residual mean+std flags AI's motion
                  # incoherence (flicker/drift) -- the best video-specific artifact cue.
                  # Gated -> old 2-branch checkpoints still load. frequency_branch is
                  # also available (set True) for ablation.
                  "motion_branch": True, "motion_dim": 128, "frequency_branch": False, "logit_fusion": True},
        # batch_videos = windows per step (each = window_len frames through the
        # backbone). 4 fits a P100/T4 16GB with AMP; if phase 2 OOMs, use 2.
        # dropout 0.4 + weight_decay 4e-4 (up from 0.3 / 1e-4): the last run
        # peaked epoch 7 then overfit epoch 8; stronger regularization + the
        # heavier aug above should push the peak higher before it overfits.
        "train": {"batch_videos": 4, "num_workers": 2, "epochs_head": 3, "epochs_finetune": 12,
                  "lr_head": 1e-3, "lr_finetune_head": 1e-4, "lr_backbone": 1e-5,
                  "weight_decay": 4e-4, "unfreeze_blocks": 3, "early_stop_patience": 4,
                  "aux_loss_weight": 0.3, "motion_aux_weight": 0.6, "use_weighted_sampler": False, "amp": True},
        "augment": {"aspect_jitter": True, "portrait_squash": True, "horizontal_flip": True,
                    "lighting": True, "resolution": True, "compression": True, "blur": True, "noise": True, "overlays": True},
        "inference": {"frames_per_video": 32, "cam_threshold": 0.60, "max_boxes": 3,
                      "verdict_real_below": 35, "verdict_fake_above": 65},
        "smoke_videos_per_source": None,
    }

import random
import numpy as np
import torch

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(CFG["seed"])
DEVICE = get_device()
print("mode:", "SMOKE TEST" if SMOKE_TEST else "FULL RUN")
print("device:", DEVICE, "| backbone:", CFG["model"]["backbone"], "| image_size:", CFG["extraction"]["image_size"])
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected -- check the Accelerator setting in Notebook Settings.")

In [ ]:
# ========================= DEPENDENCIES ======================================
!pip -q install timm albumentations
import cv2
print("deps ready")

In [ ]:
# ====================== STAGE THE FRAMES (untar OR direct) ===================
import subprocess, shutil

# Merges EVERY attached source -- tars AND pre-extracted datasets, in any
# combination. Two bugs this replaces, both of which trained on partial data
# WITHOUT any error:
#   1) `if TAR_PATHS: ... else: ...` -- with one tar + one pre-extracted
#      dataset, the pre-extracted one was silently IGNORED (half the corpus).
#   2) multiple pre-extracted datasets hard-asserted, because frame_path is
#      relative to a single data_root.
# Fix for (2): rewrite those rows' frame_path to ABSOLUTE. On POSIX,
# Path(root) / "/abs/path" == "/abs/path", so an absolute frame_path bypasses
# data_root entirely and any number of roots can coexist.
import pandas as _pd

DATA_ROOT.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "frames").mkdir(exist_ok=True)     # may stay empty in form-B-only runs
_tables = []

for tp in TAR_PATHS:                             # form A -> relative to DATA_ROOT
    print("untar", tp.name, "...")
    subprocess.run(["tar", "-xf", str(tp), "-C", str(DATA_ROOT)], check=True)
for _c in sorted(DATA_ROOT.glob("frames_index*.csv")):
    if _c.name == "frames_index_ALL.csv":
        continue                                 # our own merged output
    _df = _pd.read_csv(_c)
    _tables.append(_df)
    print(f"  + {_c.name}: {len(_df):7d} rows  (relative to DATA_ROOT)")

for _r in EXTRACTED_ROOTS:                       # form B -> make paths ABSOLUTE
    _rp = Path(_r)
    for _c in sorted(_rp.glob("frames_index*.csv")):
        _df = _pd.read_csv(_c)
        _df["frame_path"] = [str(_rp / s) for s in _df["frame_path"]]
        _tables.append(_df)
        print(f"  + {_c.name}: {len(_df):7d} rows  (absolute -> {_rp})")

assert _tables, "no frames_index*.csv found in any attached source"
_all = _pd.concat(_tables, ignore_index=True)
# Dedup on LOGICAL frame identity, not path. One dataset can be seen twice --
# as a tar AND as Kaggle's auto-extracted copy of that same tar -- yielding
# identical frames under different paths (one relative, one absolute). Keying
# on frame_path would miss those and silently double-weight every such video.
_before = len(_all)
_key = ([c for c in ("video_id", "window_index", "pos_in_window") if c in _all.columns]
        if {"window_index", "pos_in_window"} <= set(_all.columns)
        else ["video_id", "frame_path"])
_all = _all.drop_duplicates(subset=_key, keep="first")
COMBINED_CSV = DATA_ROOT / "frames_index_ALL.csv"
_all.to_csv(COMBINED_CSV, index=False)
csv_files = [COMBINED_CSV]
CFG["paths"]["data_root"] = str(DATA_ROOT)

root = Path(CFG["paths"]["data_root"])
assert csv_files, "missing frames_index*.csv"
print(f"\nMERGED {len(_tables)} index file(s) -> {len(_all)} frame rows "
      f"({_before - len(_all)} dup rows dropped), "
      f"{_all['video_id'].nunique()} unique videos")
print(_all.groupby("source")["video_id"].nunique().to_string())
print("data_root:", root)

## Stage 2: Build the manifest

Ports `build_manifest.py`. The rules that matter:
1. **Video-level splits** -- every frame of one video shares that video's
   split (frames are near-duplicates; splitting by frame inflates metrics).
2. **Stratification** -- both `real` and `ai_generated` are spread across
   train/val/test in the configured ratios.
3. Hard-asserts no video straddles two splits.

In [ ]:
# ============================== BUILD MANIFEST ================================
import re
import pandas as pd

SPLITS = ("train", "val", "test")


def scene_key(stem, pattern):
    if pattern is None:
        return None
    m = pattern.match(stem)
    return m.group(1) if m else None


_PLATFORM_TABLE = [
    ("pexl_", "pexels_landscape"), ("pex_", "pexels_portrait"),
    ("pxbw_", "pixabay_webcam"), ("pxbf_", "pixabay_portrait"), ("pxb_", "pixabay_portrait"),
    ("ugc_", "youtube_ugc"), ("vis_", "vision_phone"),
    ("gvb_", "genvidbench"), ("avg_", "avgen"), ("dact_", "deepaction"),
]


def platform_of(stem):
    s = str(stem)
    for pref, plat in _PLATFORM_TABLE:
        if s.startswith(pref):
            return plat
    return "other"


def orient_of(w, h):
    try:
        w, h = float(w), float(h)
    except (TypeError, ValueError):
        return "unknown"
    if not (w > 0 and h > 0):
        return "unknown"
    if h > w * 1.05:
        return "portrait"
    if w > h * 1.05:
        return "landscape"
    return "square"


def assign_splits(videos: pd.DataFrame, ratios: dict, pattern, seed: int) -> dict:
    """Returns {video_id: split}, stratified so VAL and TEST each mirror the full
    mix of (class x platform x orientation). Every scene-group (a crop and its
    landscape original) is kept whole so it lands in ONE split -- no leakage."""
    videos = videos.copy()
    videos["group"] = [
        f"scene::{key}" if (key := scene_key(stem, pattern)) is not None else f"video::{vid}"
        for vid, stem in zip(videos["video_id"], videos["stem"])
    ]
    videos["platform"] = videos["stem"].map(platform_of)
    videos["orient"] = [orient_of(w, h) for w, h in zip(videos["orig_w"], videos["orig_h"])]

    groups = videos.groupby("group").agg(
        video_ids=("video_id", list),
        cls=("source", lambda s: "ai_generated" if (s != "real").any() else "real"),
        platform=("platform", "first"),
        orient=("orient", "first"),
    ).reset_index()
    groups["n_videos"] = groups["video_ids"].map(len)
    groups["stratum"] = list(zip(groups["cls"], groups["platform"], groups["orient"]))

    rng = random.Random(seed)
    assignment = {}
    for _, bucket in groups.groupby("stratum"):
        bucket = bucket.sample(frac=1.0, random_state=rng.randint(0, 2**31 - 1))
        total = int(bucket["n_videos"].sum())
        targets = {s: total * ratios[s] for s in SPLITS}
        filled = {s: 0 for s in SPLITS}
        for _, row in bucket.iterrows():
            split = max(SPLITS, key=lambda s: targets[s] - filled[s])
            filled[split] += int(row["n_videos"])
            for vid in row["video_ids"]:
                assignment[vid] = split
    return assignment


frames = pd.concat([pd.read_csv(p) for p in csv_files], ignore_index=True)
before = len(frames)
# Key on frame_path (unique per written file), NOT frame_index: overlapping
# windows on a short clip legitimately share frame_index across windows, and
# deduping on it silently deletes rows and breaks window lengths.
frames = frames.drop_duplicates(subset=["video_id", "frame_path"], keep="first")
if len(frames) != before:
    print(f"dropped {before - len(frames)} duplicate frame rows")

if SMOKE_TEST:
    n = CFG["smoke_videos_per_source"]
    print(f"SMOKE TEST: subsampling to ~{n} videos per source")
    keep_ids = (
        frames.drop_duplicates("video_id")
        .groupby("source")["video_id"]
        .apply(lambda s: s.sample(min(len(s), n), random_state=SEED))
        .explode().tolist()
    )
    frames = frames[frames["video_id"].isin(keep_ids)].reset_index(drop=True)

frames["label"] = (frames["source"] != "real").astype(int)

for _c, _default in (("orig_w", float("nan")), ("orig_h", float("nan")),
                     ("window_index", 0), ("pos_in_window", 0)):
    if _c not in frames.columns:
        frames[_c] = _default

CAP_PER_CLASS = int(CFG["splits"].get("cap_per_class", 0)) or None
TRIM_FROM = str(CFG["splits"].get("trim_from", "frames-joman"))
if CAP_PER_CLASS and not SMOKE_TEST:
    vmeta = frames.drop_duplicates("video_id")[["video_id", "source", "frame_path"]].copy()
    # Cap per BINARY class (real vs ai), not per fine-grained source -- collapse
    # every non-"real" source to one "ai_generated" bucket so cap_per_class is a
    # true per-class cap regardless of how many AI sources exist.
    vmeta["cls"] = vmeta["source"].where(vmeta["source"] == "real", "ai_generated")
    vmeta["trimmable"] = vmeta["frame_path"].astype(str).str.contains(TRIM_FROM, case=False, regex=False)
    keep_ids = set()
    for cls, grp in vmeta.groupby("cls"):
        if len(grp) <= CAP_PER_CLASS:
            keep_ids |= set(grp["video_id"]); continue
        n_drop = len(grp) - CAP_PER_CLASS
        pref = grp[grp["trimmable"]].sample(frac=1.0, random_state=SEED)
        drop_ids = list(pref["video_id"].iloc[:n_drop])
        if len(drop_ids) < n_drop:
            rest = grp[~grp["video_id"].isin(drop_ids)].sample(frac=1.0, random_state=SEED)
            drop_ids += list(rest["video_id"].iloc[:n_drop - len(drop_ids)])
        keep_ids |= (set(grp["video_id"]) - set(drop_ids))
        n_from_trim = int(pref["video_id"].iloc[:n_drop].isin(drop_ids).sum())
        print(f"trim {cls}: dropped {len(drop_ids)} to reach {CAP_PER_CLASS} "
              f"({n_from_trim} from '{TRIM_FROM}', {len(drop_ids) - n_from_trim} elsewhere)")
    frames = frames[frames["video_id"].isin(keep_ids)].reset_index(drop=True)

regex = CFG["splits"].get("scene_regex")
pattern = re.compile(regex) if regex else None
ratios = {s: float(CFG["splits"][s]) for s in SPLITS}
assert abs(sum(ratios.values()) - 1.0) < 1e-6, f"split ratios must sum to 1.0, got {ratios}"

videos = frames.drop_duplicates("video_id")[["video_id", "stem", "source", "orig_w", "orig_h"]]
assignment = assign_splits(videos, ratios, pattern, CFG["seed"])
frames["split"] = frames["video_id"].map(assignment)

per_video_splits = frames.groupby("video_id")["split"].nunique()
assert (per_video_splits == 1).all(), "BUG: some video_id spans more than one split"

manifest = frames[["video_id", "frame_path", "time_sec", "label", "source", "split",
                   "orig_w", "orig_h", "window_index", "pos_in_window"]]
Path(CFG["paths"]["manifest"]).parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(CFG["paths"]["manifest"], index=False)

vid_table = (frames.drop_duplicates("video_id")
            .pivot_table(index="source", columns="split", values="video_id", aggfunc="count", fill_value=0)
            .reindex(columns=list(SPLITS)))
print("Videos per source x split:\n", vid_table)

uniq = frames.drop_duplicates("video_id").copy()
uniq["platform"] = uniq["stem"].map(platform_of) if "stem" in uniq.columns     else uniq["video_id"].map(lambda v: platform_of(str(v).split("__")[1] if "__" in str(v) else v))
uniq["orient"] = [orient_of(w, h) for w, h in zip(uniq["orig_w"], uniq["orig_h"])]
print("\n=== VAL/TEST DIVERSITY -- videos per PLATFORM x split ===")
print(uniq.pivot_table(index="platform", columns="split", values="video_id",
                       aggfunc="count", fill_value=0).reindex(columns=list(SPLITS)))
print("\n=== videos per ORIENTATION x split ===")
print(uniq.pivot_table(index="orient", columns="split", values="video_id",
                       aggfunc="count", fill_value=0).reindex(columns=list(SPLITS)))

missing = [(src, sp) for src in vid_table.index for sp in SPLITS if vid_table.loc[src, sp] == 0]
for src, sp in missing:
    print(f"WARNING: source '{src}' has ZERO videos in split '{sp}'")
assert uniq["label"].nunique() == 2, (
    "Only one class present in the manifest -- both real AND ai_generated "
    "must be in the extraction output. Re-check the extraction notebook's output."
)
real_frac = float((uniq["label"] == 0).mean())
print(f"Class balance (videos): real={real_frac:.1%}, ai_generated={1 - real_frac:.1%}")
if real_frac < 0.40 or real_frac > 0.60:
    print(f"WARNING: class imbalance ({real_frac:.1%} real) -- consider train.use_weighted_sampler=True")

print(f"\nWrote {len(manifest)} rows -> {CFG['paths']['manifest']}")

## Stage 3: Dataset, model, and training

Ports `dataset.py` (live augmentation -- never cached features, so the
backbone stays fine-tunable), `model.py` (EfficientNet spatial branch), and
`train.py` (two-phase transfer learning: frozen-backbone head warmup, then
top-block fine-tuning with a low discriminative LR; early stopping on
**video-level** validation AUC, since that's what the product is graded on).

**Anti-shortcut augmentation.** `build_transforms` randomizes every
capture/transmission nuisance factor -- orientation/aspect, lighting & colour,
resolution, JPEG compression, blur, and sensor noise -- so none of them
correlates with real-vs-AI. This is what stops the model taking a shortcut like
*"low-res or compressed => fake"*; the intrinsic generation artifacts survive
these mild, probabilistic degradations while the nuisance cues are washed out.
It is the **invariance** lever. The complementary **balance** lever is data
curation: keep the same mix of orientations/resolutions/lighting across BOTH
classes so there's no bias left to exploit in the first place.

In [ ]:
# ============================== DATASET =======================================
import random
import albumentations as A
import cv2
import numpy as np
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from collections import Counter

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class RandomOverlay(A.ImageOnlyTransform):
    """Synthetic watermark/UI overlays stamped on BOTH classes -- platform UI
    lives on wild reals, generator watermarks on AI clips, so overlaid
    graphics must carry ZERO label signal. Normalized-coord params ->
    ReplayCompose repeats the SAME overlay on every frame of a window (static,
    like real watermarks). Mirrors training/src/dataset.py."""

    _CHARS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789@#_"

    def __init__(self, max_elements=3, alpha=(0.35, 0.9), p=0.3):
        super().__init__(p=p)
        self.max_elements = max_elements
        self.alpha = alpha

    def get_params(self):
        elems = []
        for _ in range(random.randint(1, self.max_elements)):
            elems.append({
                "kind": random.choice(["text", "box", "bar"]),
                "x": random.uniform(0.02, 0.72), "y": random.uniform(0.06, 0.92),
                "w": random.uniform(0.08, 0.45), "h": random.uniform(0.03, 0.12),
                "alpha": random.uniform(*self.alpha),
                "shade": random.choice([0, 255]),
                "text": "".join(random.choice(self._CHARS) for _ in range(random.randint(4, 12))),
                "scale": random.uniform(0.5, 1.4),
            })
        return {"elems": elems}

    def apply(self, img, elems=(), **params):
        out = img.copy()
        h, w = out.shape[:2]
        for e in elems:
            layer = out.copy()
            x, y = int(e["x"] * w), int(e["y"] * h)
            col = (int(e["shade"]),) * 3
            if e["kind"] == "text":
                cv2.putText(layer, e["text"], (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                            max(0.3, e["scale"] * w / 640.0), col,
                            max(1, int(round(e["scale"] * 2))), cv2.LINE_AA)
            elif e["kind"] == "box":
                cv2.rectangle(layer, (x, y),
                              (min(w - 1, x + int(e["w"] * w)), min(h - 1, y + int(e["h"] * h))),
                              col, -1)
            else:
                cv2.rectangle(layer, (0, y), (w - 1, min(h - 1, y + int(e["h"] * h))), col, -1)
            out = cv2.addWeighted(layer, e["alpha"], out, 1.0 - e["alpha"], 0)
        return out

    def get_transform_init_args_names(self):
        return ("max_elements", "alpha")


class AnamorphicSquash(A.ImageOnlyTransform):
    """Re-squash a frame's aspect in place. After resize_square, LANDSCAPE is a
    horizontally-squashed square and PORTRAIT a vertically-squashed one, so
    "works on portrait" == "invariant to squash direction". Applying BOTH
    squashes to landscape training frames keeps a portrait upload in-dist even
    with zero portrait data. log-uniform => portrait (f<1) and landscape (f>1)
    equally likely. Mirrors training/src/dataset.py."""
    def __init__(self, ratio=(0.4, 2.5), p=0.5):
        super().__init__(p=p)
        self.ratio = ratio

    def apply(self, img, f=1.0, **params):
        h, w = img.shape[:2]
        if f >= 1.0:
            iw, ih = max(1, int(round(w / f))), h
        else:
            iw, ih = w, max(1, int(round(h * f)))
        small = cv2.resize(img, (iw, ih), interpolation=cv2.INTER_AREA)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

    def get_params(self):
        lo, hi = self.ratio
        return {"f": float(np.exp(random.uniform(np.log(lo), np.log(hi))))}

    def get_transform_init_args_names(self):
        return ("ratio",)


def build_transforms(cfg, train: bool) -> A.Compose:
    # Grouped by the SHORTCUT each group defends against: randomize every
    # capture/transmission nuisance factor (orientation, lighting, resolution,
    # compression, blur, sensor noise) so none correlates with the label and
    # the model must rely on intrinsic generation artifacts (which survive
    # these mild degradations). Forensics-safe: probabilistic (p<1), moderate,
    # and blur/noise use OneOf so a frame is never destroyed by stacking.
    # Validation/inference get NONE of this. Mirrors training/src/dataset.py.
    size = cfg["extraction"]["image_size"]
    aug = cfg.get("augment", {})
    ops = [A.Resize(size, size)]
    if train:
        # -- ORIENTATION / FRAMING shortcut --
        if aug.get("aspect_jitter", True):
            ops.append(A.RandomResizedCrop(size=(size, size), scale=(0.7, 1.0), ratio=(0.5, 2.0), p=0.7))
        # anamorphic squash BOTH ways -> portrait uploads stay in-dist even with
        # all-landscape training data (complements the crop-based jitter above)
        if aug.get("portrait_squash", True):
            ops.append(AnamorphicSquash(ratio=(0.4, 2.5), p=0.5))
        if aug.get("horizontal_flip", True):
            ops.append(A.HorizontalFlip(p=0.5))
        # -- LIGHTING / COLOR / WHITE-BALANCE shortcut --
        if aug.get("lighting", True):
            ops.append(A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5))
            ops.append(A.RandomGamma(gamma_limit=(80, 120), p=0.3))
            ops.append(A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3))
        # -- RESOLUTION / QUALITY shortcut ("low-res => fake") --
        # AGGRESSIVE: the <=480p slice was 93% one class, so the model could
        # cheat on "low quality => fake". Downscaling REAL frames hard and often
        # (p=0.65, down to 0.2x) forces plenty of low-res REALs through, so
        # resolution stops predicting the label -- quality no longer matters.
        if aug.get("resolution", True):
            ops.append(A.Downscale(scale_range=(0.2, 0.95), p=0.65))
        # -- COMPRESSION / CODEC shortcut ("compressed => fake") --
        if aug.get("compression", True):
            # quality_range replaces the deprecated quality_lower/quality_upper
            # (albumentations >=1.4) -- old kwargs silently no-op on newer
            # versions and fall back to near-lossless, defeating the point.
            # Widened low end (22) to cover heavily-compressed phone uploads.
            ops.append(A.ImageCompression(quality_range=(22, 90), p=0.6))
        # -- FOCUS / SHARPNESS shortcut: both directions (real phone footage is
        # often over-sharpened, AI output soft), at most one per frame --
        if aug.get("blur", True):
            ops.append(A.OneOf([
                A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                A.MotionBlur(blur_limit=(3, 9), p=1.0),
                A.Sharpen(alpha=(0.2, 0.4), lightness=(0.8, 1.0), p=1.0),
            ], p=0.3))
        # -- DEVICE / SENSOR-NOISE shortcut (one noise only, mild) --
        if aug.get("noise", True):
            ops.append(A.OneOf([
                A.GaussNoise(std_range=(0.02, 0.10), p=1.0),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.4), p=1.0),
            ], p=0.3))
        # -- WATERMARK / UI-OVERLAY shortcut: overlays on BOTH classes --
        if aug.get("overlays", True):
            ops.append(RandomOverlay(max_elements=3, alpha=(0.35, 0.9), p=0.3))
    ops += [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return A.Compose(ops)


def build_portrait_probe_transforms(cfg):
    """Val preprocessing + a DETERMINISTIC 9:16 portrait squash. Scoring the
    landscape test set through this MEASURES portrait robustness with no
    portrait data: AUC holds => squash-invariant => portrait handled."""
    size = cfg["extraction"]["image_size"]
    return A.Compose([A.Resize(size, size), AnamorphicSquash(ratio=(0.5625, 0.5625), p=1.0),
                      A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()])


class FrameDataset(Dataset):
    def __init__(self, manifest, transforms, data_root, with_meta=False):
        self.df = manifest.reset_index(drop=True)
        self.transforms = transforms
        self.data_root = Path(data_root)
        self.with_meta = with_meta

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        full_path = self.data_root / row["frame_path"]
        bgr = cv2.imread(str(full_path))
        if bgr is None:
            raise FileNotFoundError(f"Missing frame on disk: {full_path}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        image = self.transforms(image=rgb)["image"]
        label = torch.tensor(float(row["label"]), dtype=torch.float32)
        if self.with_meta:
            return image, label, str(row["video_id"]), str(row["source"])
        return image, label


# ---------------- HYBRID: windows, not frames --------------------------------
# One item = one CONTIGUOUS window (T, C, H, W). The augmentation for a window
# is sampled ONCE and REPLAYED identically on every frame -- per-frame random
# params would inject fake temporal flicker that swamps the real motion signal
# the ConvLSTM must learn. Mirrors training/src/dataset.py.


def build_window_transforms(cfg, train):
    return A.ReplayCompose(build_transforms(cfg, train).transforms)


class WindowDataset(Dataset):
    """One item per (video_id, window_index). Keeps only COMPLETE windows
    (the modal frame count) -- the ConvLSTM needs a fixed sequence length."""

    def __init__(self, manifest, transforms, data_root, with_meta=False):
        df = manifest.reset_index(drop=True)
        expected = int(df.groupby(["video_id", "window_index"]).size().mode().iloc[0])
        groups, dropped = [], 0
        for (vid, _w), g in df.groupby(["video_id", "window_index"]):
            if len(g) != expected:
                dropped += 1
                continue
            g = g.sort_values("pos_in_window")
            groups.append({"video_id": str(vid), "source": str(g["source"].iloc[0]),
                           "label": float(g["label"].iloc[0]),
                           "frame_paths": g["frame_path"].tolist()})
        if dropped:
            print(f"WindowDataset: dropped {dropped} incomplete windows (expected {expected} frames)")
        self.window_len = expected
        self.groups = groups
        self.transforms = transforms
        self.data_root = Path(data_root)
        self.with_meta = with_meta

    def __len__(self):
        return len(self.groups)

    def _read(self, rel):
        full = self.data_root / rel
        bgr = cv2.imread(str(full))
        if bgr is None:
            raise FileNotFoundError(f"Missing frame on disk: {full}")
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        g = self.groups[idx]
        first = self.transforms(image=self._read(g["frame_paths"][0]))
        frames = [first["image"]]
        replay = first["replay"]     # one param draw, replayed on every frame
        for rel in g["frame_paths"][1:]:
            frames.append(A.ReplayCompose.replay(replay, image=self._read(rel))["image"])
        window = torch.stack(frames, dim=0)          # (T, C, H, W)
        label = torch.tensor(g["label"], dtype=torch.float32)
        if self.with_meta:
            return window, label, g["video_id"], g["source"]
        return window, label


def make_window_dataloaders(cfg, manifest):
    tcfg = cfg["train"]
    data_root = cfg["paths"]["data_root"]
    train_df = manifest[manifest["split"] == "train"]
    val_df = manifest[manifest["split"] == "val"]

    train_ds = WindowDataset(train_df, build_window_transforms(cfg, train=True), data_root)
    val_ds = WindowDataset(val_df, build_window_transforms(cfg, train=False), data_root, with_meta=True)

    sampler, shuffle = None, True
    if tcfg.get("use_weighted_sampler", False):
        counts = Counter(g["label"] for g in train_ds.groups)
        weights = np.array([1.0 / counts[g["label"]] for g in train_ds.groups], dtype=np.float64)
        sampler = WeightedRandomSampler(torch.from_numpy(weights), num_samples=len(weights), replacement=True)
        shuffle = False

    common = dict(batch_size=tcfg.get("batch_videos", 4), num_workers=tcfg["num_workers"],
                  pin_memory=torch.cuda.is_available(), persistent_workers=tcfg["num_workers"] > 0)
    train_loader = DataLoader(train_ds, shuffle=shuffle, sampler=sampler, drop_last=True, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)
    return train_loader, val_loader


manifest_df = pd.read_csv(CFG["paths"]["manifest"])
for split in ("train", "val"):
    classes = manifest_df.loc[manifest_df["split"] == split, "label"].nunique()
    assert classes == 2, f"split '{split}' has {classes} class(es) -- cannot train"

train_loader, val_loader = make_window_dataloaders(CFG, manifest_df)
T = train_loader.dataset.window_len
print(f"train windows: {len(train_loader.dataset)} | val windows: {len(val_loader.dataset)} "
      f"| window_len T={T} ({T} contiguous frames per item)")

In [ ]:
# ============================== MODEL ==========================================
import timm
from torch import nn


class SpatialBranch(nn.Module):
    """timm backbone -> global avg pool -> dropout -> linear -> 1 logit."""

    def __init__(self, backbone, pretrained, dropout):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="avg")
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 1))

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_top_blocks(self, n_blocks):
        """Unfreezes the last n_blocks stages + the trailing conv head/bn --
        low-level filters (edges, textures) transfer fine from ImageNet;
        later stages hold the task-specific features we adapt."""
        blocks = getattr(self.backbone, "blocks", None)
        if blocks is not None:
            for stage in list(blocks)[-n_blocks:]:
                for p in stage.parameters():
                    p.requires_grad = True
        for attr in ("conv_head", "bn2"):
            module = getattr(self.backbone, attr, None)
            if module is not None:
                for p in module.parameters():
                    p.requires_grad = True

    def backbone_trainable_params(self):
        return [p for p in self.backbone.parameters() if p.requires_grad]


class ConvLSTMCell(nn.Module):
    """LSTM cell with CONVOLUTIONAL gates over spatial feature maps -- state
    (h, c) keeps its spatial layout, so temporal reasoning happens per spatial
    region (localized flicker), not on a blurred global vector."""

    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.gates = nn.Conv2d(in_channels + hidden_channels, 4 * hidden_channels,
                               kernel_size, padding=kernel_size // 2)

    def forward(self, x, h, c):
        i, f, g, o = self.gates(torch.cat([x, h], dim=1)).chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        c = f * c + i * torch.tanh(g)
        h = o * torch.tanh(c)
        return h, c


class TemporalBranch(nn.Module):
    """ConvLSTM over the per-frame backbone feature maps of one window."""

    def __init__(self, in_channels, hidden_channels=256, kernel_size=3, dropout=0.3):
        super().__init__()
        self.cell = ConvLSTMCell(in_channels, hidden_channels, kernel_size)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_channels, 1))

    def forward(self, feats):                        # (B, T, F, h, w)
        b, t, _, hh, ww = feats.shape
        h = feats.new_zeros(b, self.cell.hidden_channels, hh, ww)
        c = feats.new_zeros(b, self.cell.hidden_channels, hh, ww)
        for step in range(t):
            h, c = self.cell(feats[:, step], h, c)
        pooled = h.mean(dim=(2, 3))
        return pooled, self.head(pooled).squeeze(1)


class FrequencyBranch(nn.Module):
    """Per-frame FFT log-magnitude -> small CNN -> feature + logit. Diffusion/GAN
    upsamplers leave periodic grid patterns in the FREQUENCY domain, invisible
    in pixel space and content-independent (they transfer across generators) --
    a real generation artifact, not a shortcut. Third axis alongside spatial
    (texture) + temporal (motion); tiny (3-conv CNN over the spectrum)."""
    def __init__(self, feat_dim=128, dropout=0.3):
        super().__init__()
        self.feat_dim = feat_dim
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, feat_dim, 3, stride=2, padding=1), nn.BatchNorm2d(feat_dim), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 1))
    @staticmethod
    def _spectrum(x):
        # FFT runs in float32 (unsupported/unstable under AMP half).
        gray = x.float().mean(dim=1, keepdim=True)
        fft = torch.fft.fftshift(torch.fft.fft2(gray), dim=(-2, -1))
        logmag = torch.log1p(fft.abs())
        mu = logmag.mean(dim=(-2, -1), keepdim=True)
        sd = logmag.std(dim=(-2, -1), keepdim=True) + 1e-6
        return (logmag - mu) / sd
    def forward(self, frames):
        feat = self.cnn(self._spectrum(frames)).flatten(1)
        return feat, self.head(feat).squeeze(1)


class MotionBranch(nn.Module):
    """Motion-incoherence branch: from luminance residuals it builds four CONTENT-
    NORMALIZED temporal statistic maps (drift, flicker=temporal std, energy, jerk)
    and reads them with a CNN. Keys on the PATTERN of motion inconsistency (AI
    flicker/warp), not how much the scene moves; maps keep WHERE the artifacts are."""
    def __init__(self, feat_dim=128, dropout=0.3):
        super().__init__()
        self.feat_dim = feat_dim
        self.out_dim = feat_dim
        # GroupNorm: time is pooled into maps before the CNN, so batch here is ~B (too small for BN).
        self.cnn = nn.Sequential(
            nn.Conv2d(4, 32, 3, stride=2, padding=1), nn.GroupNorm(8, 32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.GroupNorm(8, 64), nn.ReLU(inplace=True),
            nn.Conv2d(64, feat_dim, 3, stride=2, padding=1), nn.GroupNorm(8, feat_dim), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 1))
    def forward(self, x):
        gray = x.float().mean(dim=2)
        d1 = gray[:, 1:] - gray[:, :-1]
        scale = d1.abs().mean(dim=(1, 2, 3), keepdim=True) + 1e-4
        d1 = d1 / scale
        drift = d1.mean(dim=1, keepdim=True)
        flicker = d1.std(dim=1, keepdim=True)
        energy = d1.abs().mean(dim=1, keepdim=True)
        jerk = (d1[:, 1:] - d1[:, :-1]).abs().mean(dim=1, keepdim=True)
        stat = torch.cat([drift, flicker, energy, jerk], dim=1)
        feat = self.cnn(stat).flatten(1)
        return feat, self.head(feat).squeeze(1)


class ChronosHybrid(nn.Module):
    """EfficientNet + ConvLSTM (+ optional Frequency) + fusion, trained JOINTLY.
    Input: windows (B, T, C, H, W). Mirrors training/src/model.py -- same
    attribute names, so the checkpoint loads into the local ChronosHybrid
    without remapping. The frequency branch is gated by model.frequency_branch
    so pre-frequency (2-branch) checkpoints still load."""

    def __init__(self, cfg):
        super().__init__()
        mcfg = cfg["model"]
        lcfg = mcfg.get("convlstm", {})
        self.spatial = SpatialBranch(mcfg["backbone"], mcfg["pretrained"], mcfg["dropout"])
        feat_dim = self.spatial.backbone.num_features
        hidden = int(lcfg.get("hidden_channels", 256))
        self.temporal = TemporalBranch(feat_dim, hidden, int(lcfg.get("kernel_size", 3)), mcfg["dropout"])
        drop = mcfg["dropout"]
        fusion_in = feat_dim + hidden
        self.use_frequency = bool(mcfg.get("frequency_branch", False))
        if self.use_frequency:
            self.frequency = FrequencyBranch(int(mcfg.get("freq_dim", 128)), drop)
            fusion_in += self.frequency.feat_dim
        self.use_motion = bool(mcfg.get("motion_branch", False))
        if self.use_motion:
            self.motion = MotionBranch(int(mcfg.get("motion_dim", 128)), drop)
            fusion_in += self.motion.out_dim
        fusion_hidden = int(mcfg.get("fusion_hidden", 256))
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, fusion_hidden), nn.ReLU(inplace=True),
            nn.Dropout(mcfg["dropout"]), nn.Linear(fusion_hidden, 1))
        # Logit-level fusion gate: learned softmax weighting over [MLP-fusion +
        # each branch logit] so the model can fall back to the strongest branch
        # (fixes fused < best-branch). Equal init => starts as a mean ensemble.
        self.use_logit_fusion = bool(mcfg.get("logit_fusion", False))
        if self.use_logit_fusion:
            n_members = 3 + int(self.use_frequency) + int(self.use_motion)
            self.fusion_gate = nn.Parameter(torch.zeros(n_members))

    def _backbone_frozen(self):
        return not any(p.requires_grad for p in self.spatial.backbone.parameters())

    def forward(self, x):                             # (B, T, C, H, W)
        b, t, c, h, w = x.shape
        flat = x.reshape(b * t, c, h, w)
        # Phase 1 trains only heads/ConvLSTM: frozen backbone -> no_grad is
        # safe and slashes activation memory.
        if self._backbone_frozen():
            with torch.no_grad():
                fmaps = self.spatial.backbone.forward_features(flat)
        else:
            fmaps = self.spatial.backbone.forward_features(flat)
        fh, fw = fmaps.shape[-2], fmaps.shape[-1]
        fdim = fmaps.shape[1]
        pooled = fmaps.mean(dim=(2, 3))                                        # (B*T, F)
        spatial_logit = self.spatial.head(pooled).squeeze(1).reshape(b, t).mean(dim=1)
        temporal_feat, temporal_logit = self.temporal(fmaps.reshape(b, t, fdim, fh, fw))
        spatial_feat = pooled.reshape(b, t, fdim).mean(dim=1)
        feats = [spatial_feat, temporal_feat]
        out = {"spatial_logit": spatial_logit, "temporal_logit": temporal_logit}
        if self.use_frequency:
            ff_flat, fl_flat = self.frequency(flat)
            feats.append(ff_flat.reshape(b, t, -1).mean(dim=1))
            out["frequency_logit"] = fl_flat.reshape(b, t).mean(dim=1)
        if self.use_motion:
            motion_feat, motion_logit = self.motion(x)
            feats.append(motion_feat)
            out["motion_logit"] = motion_logit
        fusion_logit = self.fusion(torch.cat(feats, dim=1)).squeeze(1)
        if self.use_logit_fusion:
            members = [fusion_logit, spatial_logit, temporal_logit]
            if self.use_frequency:
                members.append(out["frequency_logit"])
            if self.use_motion:
                members.append(out["motion_logit"])
            w = torch.softmax(self.fusion_gate, dim=0)
            out["logit"] = sum(wi * m for wi, m in zip(w, members))
        else:
            out["logit"] = fusion_logit
        return out

    def freeze_backbone(self):
        self.spatial.freeze_backbone()

    def unfreeze_top_blocks(self, n_blocks):
        self.spatial.unfreeze_top_blocks(n_blocks)

    def backbone_trainable_params(self):
        return self.spatial.backbone_trainable_params()

    def head_params(self):
        params = list(self.spatial.head.parameters()) + list(self.temporal.parameters())
        if self.use_frequency:
            params += list(self.frequency.parameters())
        if self.use_motion:
            params += list(self.motion.parameters())
        params += list(self.fusion.parameters())
        if self.use_logit_fusion:
            params.append(self.fusion_gate)
        return params


model = ChronosHybrid(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
n_temporal = sum(p.numel() for p in model.temporal.parameters())
n_fusion = sum(p.numel() for p in model.fusion.parameters())
print(f"HYBRID ready: {CFG['model']['backbone']} + ConvLSTM({CFG['model']['convlstm']['hidden_channels']}ch) "
      f"+ fusion | {n_params:,} params (temporal {n_temporal:,}, fusion {n_fusion:,})")

In [ ]:
# ============================== EVAL HELPERS (used by train + final eval) =====
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)


def safe_auc(y_true, y_prob) -> float:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, np.asarray(y_prob)))


@torch.no_grad()
def collect_window_probs(model, loader, device, criterion=None):
    """One record per WINDOW: fused prob (the model's answer) plus per-branch
    probs, so the final eval can report fused vs spatial vs temporal AUC."""
    model.eval()
    records = []
    total_loss, seen = 0.0, 0
    for windows, labels, video_ids, sources in loader:
        windows = windows.to(device, non_blocking=True)
        labels_dev = labels.to(device, non_blocking=True)
        out = model(windows)
        if criterion is not None:
            total_loss += criterion(out["logit"], labels_dev).item() * len(labels)
            seen += len(labels)
        fused = torch.sigmoid(out["logit"]).cpu().numpy()
        spat = torch.sigmoid(out["spatial_logit"]).cpu().numpy()
        temp = torch.sigmoid(out["temporal_logit"]).cpu().numpy()
        freq = torch.sigmoid(out["frequency_logit"]).cpu().numpy() if "frequency_logit" in out else None
        mot = torch.sigmoid(out["motion_logit"]).cpu().numpy() if "motion_logit" in out else None
        for i, (vid, source, label, p, ps, pt) in enumerate(
                zip(video_ids, sources, labels.numpy(), fused, spat, temp)):
            rec = {"video_id": vid, "source": source, "label": int(label),
                   "prob": float(p), "prob_spatial": float(ps), "prob_temporal": float(pt)}
            if freq is not None: rec["prob_frequency"] = float(freq[i])
            if mot is not None:  rec["prob_motion"] = float(mot[i])
            records.append(rec)
    return pd.DataFrame.from_records(records), (total_loss / seen if seen else float("nan"))


def aggregate_videos(frame_df):
    agg = dict(label=("label", "first"), source=("source", "first"),
               mean_prob=("prob", "mean"), max_prob=("prob", "max"))
    for extra in ("prob_spatial", "prob_temporal", "prob_frequency", "prob_motion"):
        if extra in frame_df.columns:
            agg[f"mean_{extra}"] = (extra, "mean")
    return frame_df.groupby("video_id").agg(**agg).reset_index()


def compute_metrics(video_df, threshold=0.5):
    y_true = video_df["label"].to_numpy()
    y_prob = video_df["mean_prob"].to_numpy()
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "n_videos": int(len(video_df)), "auc": safe_auc(y_true, y_prob),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }


def per_source_metrics(video_df):
    out = {}
    real = video_df[video_df["source"] == "real"]
    for source in sorted(s for s in video_df["source"].unique() if s != "real"):
        subset = pd.concat([real, video_df[video_df["source"] == source]])
        out[source] = compute_metrics(subset)
    return out


# ---- sliced eval: is orientation/resolution acting as a shortcut? ------------
# Two symptoms: (1) class skew INSIDE a slice (e.g. all portrait = fake) -> the
# factor correlates with the label, model can cheat; (2) an AUC gap ACROSS
# slices -> model didn't learn a robust cue there. Needs orig_w/orig_h from
# extraction. Mirrors training/src/evaluate.py.
def slice_of(w, h):
    try:
        w, h = float(w), float(h)
    except (TypeError, ValueError):
        return "unknown", "unknown"
    if w != w or h != h or w <= 0 or h <= 0:      # NaN / missing / zero
        return "unknown", "unknown"
    r = w / h
    orient = "portrait" if r < 0.95 else "landscape" if r > 1.05 else "square"
    short = min(w, h)      # short side -> orientation-agnostic "p" bucket (upper bounds)
    res = ("<=360p" if short <= 360 else "<=480p" if short <= 480 else
           "<=720p" if short <= 720 else "<=1080p" if short <= 1080 else ">1080p")
    return orient, res


def add_slice_columns(video_df, meta_df):
    meta = meta_df.drop_duplicates("video_id")[["video_id", "orig_w", "orig_h"]]
    out = video_df.merge(meta, on="video_id", how="left")
    slices = [slice_of(w, h) for w, h in zip(out["orig_w"], out["orig_h"])]
    out["orientation"] = [s[0] for s in slices]
    out["resolution"] = [s[1] for s in slices]
    return out


def sliced_metrics(video_df, by):
    out = {}
    for value, grp in video_df.groupby(by):
        y = grp["label"].to_numpy()
        n_real, n_fake = int((y == 0).sum()), int((y == 1).sum())
        m = compute_metrics(grp)
        out[str(value)] = {"n_videos": int(len(grp)), "n_real": n_real, "n_fake": n_fake,
                           "fake_rate": round(n_fake / max(len(grp), 1), 3),
                           "auc": m["auc"], "accuracy": m["accuracy"]}
    return out


def format_slice_report(name, sliced):
    lines = [f"=== AUC by {name} ===",
             f"{'slice':>12} {'n':>5} {'real':>5} {'fake':>5} {'fake%':>6} {'auc':>7} {'acc':>7}"]
    aucs = []
    for value, m in sorted(sliced.items()):
        auc = m["auc"]
        if auc == auc and m["n_real"] > 0 and m["n_fake"] > 0:
            aucs.append(auc)
        auc_s = f"{auc:.3f}" if auc == auc else "n/a"
        lines.append(f"{value:>12} {m['n_videos']:>5} {m['n_real']:>5} {m['n_fake']:>5} "
                     f"{m['fake_rate'] * 100:>5.0f}% {auc_s:>7} {m['accuracy']:>7.3f}")
    for value, m in sorted(sliced.items()):
        if value == "unknown" or m["n_videos"] < 5:
            continue
        if m["fake_rate"] >= 0.85 or m["fake_rate"] <= 0.15:
            lines.append(f"  [SHORTCUT RISK] {name}={value} is {m['fake_rate'] * 100:.0f}% one class -- "
                         f"{name} correlates with the label here; the model can cheat on it. "
                         f"Add the missing class for this {name}.")
    if len(aucs) >= 2:
        gap = max(aucs) - min(aucs)
        if gap >= 0.10:
            lines.append(f"  [WEAK SLICE] AUC spans {min(aucs):.3f}-{max(aucs):.3f} across {name} "
                         f"(gap {gap:.3f}) -- weaker on some {name} values; add data / stronger aug there.")
        else:
            lines.append(f"  [OK] AUC stable across {name} (gap {gap:.3f}) -- no {name} shortcut evident.")
    return "\n".join(lines)


# ---- per-GENERATOR eval: WHICH models does the detector actually beat? ------
# The diversity batch tags the generator in the filename (avg_<model>_<hash>),
# which survives inside video_id (<source>__<stem>__<hash8>). Untagged AI
# reports as "legacy_pool". Mirrors training/src/evaluate.py.
def generator_of(video_id):
    parts = str(video_id).split("__")
    if len(parts) < 3:
        return None
    stem = "__".join(parts[1:-1])
    if stem.startswith("avg_"):
        tail = stem[4:]
        return tail.rsplit("_", 1)[0] if "_" in tail else tail   # drop the hash
    return "legacy_pool"


def per_generator_metrics(video_df):
    real = video_df[video_df["label"] == 0]
    fakes = video_df[video_df["label"] == 1].copy()
    if fakes.empty or real.empty:
        return {}
    fakes["generator"] = [generator_of(v) or "unparsed" for v in fakes["video_id"]]
    out = {}
    for gen, grp in fakes.groupby("generator"):
        m = compute_metrics(pd.concat([real, grp]))
        out[str(gen)] = {"n_fake": int(len(grp)), "n_real": int(len(real)),
                         "auc": m["auc"], "accuracy": m["accuracy"], "recall_on_fakes": m["recall"]}
    return out


def real_source_of(video_id):
    """Which collection a REAL video came from (deterministic stem prefixes
    from the batch notebooks). Mirrors training/src/evaluate.py."""
    parts = str(video_id).split("__")
    stem = "__".join(parts[1:-1]) if len(parts) >= 3 else str(video_id)
    for prefix, tag in (("ugc_", "youtube_ugc"), ("vis_", "vision_devices"),
                        ("pexl_", "pexels"), ("pex_", "pexels"), ("dact_", "deepaction")):
        if stem.startswith(prefix):
            return tag
    return "stock_pool"


def per_real_source_fpr(video_df, threshold):
    reals = video_df[video_df["label"] == 0].copy()
    if reals.empty:
        return {}
    reals["real_source"] = [real_source_of(v) for v in reals["video_id"]]
    out = {}
    for src, grp in reals.groupby("real_source"):
        fp = int((grp["mean_prob"] >= threshold).sum())
        out[str(src)] = {"n": int(len(grp)), "false_positives": fp, "fpr": round(fp / len(grp), 4)}
    return out


def format_real_source_report(per_src, threshold):
    lines = [f"=== false-positive rate per REAL source (at t*={threshold:.3f}) ===",
             f"{'real source':>16} {'n':>6} {'flagged':>8} {'FPR':>7}"]
    for src, m in sorted(per_src.items(), key=lambda kv: -kv[1]["fpr"]):
        note = "  <- small n, noisy" if m["n"] < 15 else ""
        lines.append(f"{src:>16} {m['n']:>6} {m['false_positives']:>8} {m['fpr']*100:>6.1f}%{note}")
    lines.append("  (goal: wild sources' FPR approaching stock_pool's)")
    return "\n".join(lines)


def format_generator_report(per_gen):
    lines = ["=== AUC by generator (all real vs each generator's fakes) ===",
             f"{'generator':>24} {'n_fake':>7} {'auc':>7} {'acc':>7} {'recall':>7}"]
    scored = []
    for gen, m in sorted(per_gen.items(), key=lambda kv: (kv[1]["auc"] != kv[1]["auc"], kv[1]["auc"])):
        auc_s = f"{m['auc']:.3f}" if m["auc"] == m["auc"] else "n/a"
        flag = "  <- small n, noisy" if m["n_fake"] < 15 else ""
        lines.append(f"{gen:>24} {m['n_fake']:>7} {auc_s:>7} {m['accuracy']:>7.3f} "
                     f"{m['recall_on_fakes']:>7.3f}{flag}")
        if m["auc"] == m["auc"] and m["n_fake"] >= 15:
            scored.append((gen, m["auc"]))
    if scored:
        worst = min(scored, key=lambda x: x[1])
        lines.append(f"  weakest generator: {worst[0]} (AUC {worst[1]:.3f}) -- "
                     f"add data there or accept and report it.")
    return "\n".join(lines)


def validate(model, loader, criterion, device):
    window_df, val_loss = collect_window_probs(model, loader, device, criterion)
    video_df = aggregate_videos(window_df)
    return {"val_loss": val_loss,
            "val_window_auc": safe_auc(window_df["label"], window_df["prob"]),
            "val_video_auc": safe_auc(video_df["label"], video_df["mean_prob"]),
            "val_spatial_auc": safe_auc(video_df["label"], video_df["mean_prob_spatial"]),
            "val_temporal_auc": safe_auc(video_df["label"], video_df["mean_prob_temporal"])}

print("eval helpers ready")

In [ ]:
# ============================== TRAIN ==========================================
import json
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

RUN_DIR = OUTPUTS_DIR / f"run_{time.strftime('%Y%m%d_%H%M%S')}_{'smoke' if SMOKE_TEST else 'v1'}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run dir:", RUN_DIR)

# WALL-CLOCK BUDGET: Kaggle kills a commit at 12h and a killed commit publishes
# NOTHING -- not even the best.pt already sitting on disk. At 12k videos the
# hybrid takes ~1.3-1.7h/epoch, so 15 epochs cannot fit. When the budget is
# reached, training stops AFTER the current epoch (best.pt is already saved on
# every improvement) and the notebook proceeds to the test evaluation, so the
# commit ALWAYS completes with a usable model + metrics.
# PREDICTIVE guard. The old check ran AFTER each epoch ("have I exceeded 9.5h?"),
# so it could start an epoch at 9.1h and finish at 11.4h -- leaving almost no
# room for calibration + test eval before the 12h kill. At 22k videos an epoch
# is ~2.3h, so that overshoot is a full epoch wide.
# Now: before starting another epoch we ask "will this epoch AND the final eval
# still fit?" using the measured average epoch time.
TIME_BUDGET_HOURS = 11.0     # total wall-clock this cell may consume (12h kill - margin)
EVAL_RESERVE_HOURS = 0.75    # held back for calibration + test eval + saving
TRAIN_START = time.time()


def configure_phase(model, cfg, phase):
    """Phase 1: frozen backbone -> train ALL heads (spatial head + ConvLSTM +
    fusion). Phase 2: unfreeze top backbone blocks at a low discriminative LR;
    heads keep a higher one."""
    tcfg = cfg["train"]
    if phase == 1:
        model.freeze_backbone()
        optimizer = torch.optim.AdamW(model.head_params(), lr=tcfg["lr_head"], weight_decay=tcfg["weight_decay"])
        scheduler = None
    else:
        model.freeze_backbone()
        model.unfreeze_top_blocks(tcfg["unfreeze_blocks"])
        optimizer = torch.optim.AdamW(
            [{"params": model.backbone_trainable_params(), "lr": tcfg["lr_backbone"]},
             {"params": model.head_params(), "lr": tcfg["lr_finetune_head"]}],
            weight_decay=tcfg["weight_decay"],
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=tcfg["epochs_finetune"])
    return optimizer, scheduler


def train_one_epoch(model, loader, optimizer, criterion, device, scaler, use_amp, aux_w, motion_aux_w):
    """BCE(fused) + aux_w*(spatial+temporal[+freq]) + motion_aux_w*motion. Motion
    gets a heavier aux weight -- it's the weakest branch, so a stronger direct
    signal keeps its head discriminative instead of collapsing to ~0.5."""
    model.train()
    total_loss, correct, seen = 0.0, 0, 0
    for windows, labels in tqdm(loader, desc="train", leave=False):
        windows, labels = windows.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(windows)
            aux = (criterion(out["spatial_logit"], labels)
                   + criterion(out["temporal_logit"], labels))
            if "frequency_logit" in out:
                aux = aux + criterion(out["frequency_logit"], labels)
            loss = criterion(out["logit"], labels) + aux_w * aux
            if "motion_logit" in out:
                loss = loss + motion_aux_w * criterion(out["motion_logit"], labels)
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += ((torch.sigmoid(out["logit"]) > 0.5).float() == labels).sum().item()
        seen += len(labels)
    return total_loss / max(seen, 1), correct / max(seen, 1)


def save_curves(history, out_path):
    epochs = [h["epoch"] for h in history]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(epochs, [h["train_loss"] for h in history], label="train loss")
    ax1.plot(epochs, [h["val_loss"] for h in history], label="val loss")
    ax1.set_xlabel("epoch"); ax1.set_title("Loss (gap = overfitting signal)"); ax1.legend()
    ax2.plot(epochs, [h["train_acc"] for h in history], label="train acc")
    ax2.plot(epochs, [h["val_video_auc"] for h in history], label="val video AUC")
    ax2.set_xlabel("epoch"); ax2.set_title("Accuracy / AUC"); ax2.legend()
    fig.tight_layout(); fig.savefig(out_path, dpi=120); plt.close(fig)


criterion = nn.BCEWithLogitsLoss()
use_amp = CFG["train"].get("amp", True) and DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda") if use_amp else None

epochs_head = CFG["train"]["epochs_head"]
total_epochs = epochs_head + CFG["train"]["epochs_finetune"]
patience = CFG["train"]["early_stop_patience"]

best_auc, bad_epochs = float("-inf"), 0
history = []
optimizer = scheduler = None
start_epoch = 0

# ---- RESUME (same data, same training -- just continuing an interrupted run).
# To use: Add Input -> Your Work -> the PREVIOUS training version; its last.pt
# is found here and training continues from the next epoch with weights,
# history, best-AUC, and (same-phase) optimizer state restored. Splits are
# seed-deterministic, so train/val/test are identical across commits -- no
# leakage. Nothing attached -> trains from scratch. NOTE: this is only for
# continuing THIS dataset; adding NEW data still means a fresh run.
RESUME = True   # set False to force from-scratch even if a last.pt is attached
# EVAL_ONLY: skip training entirely, load the attached best.pt, and run just the
# calibration + test evaluation (~40 min instead of a 12h commit). Use this when
# a training commit ran out of time before writing metrics_test.json.
EVAL_ONLY = False   # <<< EDIT ME
import glob as _glob
import zipfile as _zipf
_resume = None


def _find_checkpoint(name: str):
    """Locate 'name'.pt under /kaggle/input, tolerating Kaggle's auto-extraction.
    A .pt IS a zip archive, so Kaggle may publish it as a FOLDER (name/data.pkl
    + name/data/*) with no .pt file at all. Rebuild a loadable .pt from that."""
    hits = [p for p in sorted(_glob.glob(f"/kaggle/input/**/{name}.pt", recursive=True))]
    hits = [p for p in hits if "smoke" not in p] or hits
    if hits:
        return hits[-1]
    pkls = sorted(_glob.glob(f"/kaggle/input/**/{name}/data.pkl", recursive=True))
    pkls = [p for p in pkls if "smoke" not in p] or pkls
    for pk in pkls:
        d = Path(pk).parent
        try:
            out = Path("/kaggle/tmp") / f"{name}_rebuilt.pt"
            out.parent.mkdir(parents=True, exist_ok=True)
            with _zipf.ZipFile(out, "w", _zipf.ZIP_STORED) as zf:
                for f in sorted(d.rglob("*")):
                    if f.is_file():
                        zf.write(f, f"{d.name}/{f.relative_to(d).as_posix()}")
            torch.load(out, map_location="cpu", weights_only=False)   # validate
            print(f"Kaggle had EXTRACTED {name}.pt -> rebuilt {d} into {out}")
            return str(out)
        except Exception as e:
            print(f"  could not rebuild {name} from {d}: {type(e).__name__}: {e}")
    return None
if RESUME and SMOKE_TEST:
    # A smoke run uses a small b0 model -- loading an attached full-run (b4)
    # checkpoint into it would crash on mismatched shapes, and resuming into a
    # plumbing check is never wanted anyway.
    print("SMOKE_TEST -> resume disabled for this run")
    RESUME = False
if EVAL_ONLY:
    # Load the attached best.pt, copy it into this run dir, and skip training
    # entirely -- the cells below (calibration + test eval) then run against it.
    _bp = _find_checkpoint("best")
    assert _bp, ("EVAL_ONLY needs a best.pt attached (the training run's output "
                 "dataset). Nothing found under /kaggle/input.")
    _b = torch.load(_bp, map_location=DEVICE, weights_only=False)
    model.load_state_dict(_b["model"])
    best_auc = float(_b.get("best_metric", float("-inf")))
    history = list(_b.get("history", []))
    shutil.copy(_bp, RUN_DIR / "best.pt")
    total_epochs = start_epoch          # empty range -> training loop is skipped
    print(f"EVAL_ONLY -> loaded {_bp} (epoch {_b.get('epoch')}, best AUC {best_auc:.4f})")
    print("            skipping training; running calibration + test eval only.")
elif RESUME:
    _ck = _find_checkpoint("last")
    if _ck:
        _resume = torch.load(_ck, map_location=DEVICE, weights_only=False)
        model.load_state_dict(_resume["model"])
        start_epoch = int(_resume["epoch"]) + 1
        best_auc = float(_resume.get("best_metric", float("-inf")))
        history = list(_resume.get("history", []))
        print(f"RESUMED from {_ck} -> continuing at epoch {start_epoch} "
              f"(best video AUC so far {best_auc:.4f})")
    else:
        # Loud diagnostic: show what IS attached so a missing/misnamed dataset
        # is obvious instead of silently becoming a from-scratch run.
        print("no last.pt attached -> training from scratch")
        print("DIAGNOSTIC - top-level of /kaggle/input:")
        for _p in sorted(_glob.glob("/kaggle/input/*/*"))[:25]:
            print("   ", _p)

for epoch in range(start_epoch, total_epochs):
    phase = 1 if epoch < epochs_head else 2
    if optimizer is None or epoch == epochs_head:
        optimizer, scheduler = configure_phase(model, CFG, phase)
        # same-phase resume also restores optimizer/scheduler/scaler state
        if _resume is not None and epoch == start_epoch and _resume.get("phase") == phase:
            if _resume.get("optimizer"):
                optimizer.load_state_dict(_resume["optimizer"])
            if scheduler is not None and _resume.get("scheduler"):
                scheduler.load_state_dict(_resume["scheduler"])
            if scaler is not None and _resume.get("scaler"):
                scaler.load_state_dict(_resume["scaler"])
            print("  restored optimizer/scheduler/scaler state (same-phase resume)")
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Phase {phase} | trainable params: {n_trainable:,}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE,
                                            scaler, use_amp, CFG["train"]["aux_loss_weight"],
                                            CFG["train"].get("motion_aux_weight", CFG["train"]["aux_loss_weight"]))
    val_metrics = validate(model, val_loader, criterion, DEVICE)
    if scheduler is not None:
        scheduler.step()

    row = {"epoch": epoch, "phase": phase, "train_loss": round(train_loss, 5), "train_acc": round(train_acc, 5),
          "lr": optimizer.param_groups[0]["lr"],
          **{k: (round(v, 5) if v == v else None) for k, v in val_metrics.items()}}
    history.append(row)
    print(f"epoch {epoch:02d} [phase {phase}] train_loss={train_loss:.4f} acc={train_acc:.3f} | "
         f"val_loss={val_metrics['val_loss']:.4f} video_auc={val_metrics['val_video_auc']:.4f} "
         f"(spatial {val_metrics['val_spatial_auc']:.4f} / temporal {val_metrics['val_temporal_auc']:.4f})")

    state = dict(epoch=epoch, phase=phase, model=model.state_dict(), config=CFG,
                history=history, best_metric=best_auc,
                optimizer=optimizer.state_dict(),
                scheduler=scheduler.state_dict() if scheduler is not None else None,
                scaler=scaler.state_dict() if scaler is not None else None)
    torch.save(state, RUN_DIR / "last.pt")
    with open(RUN_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)
    save_curves(history, RUN_DIR / "curves.png")

    video_auc = val_metrics["val_video_auc"]
    if video_auc == video_auc and video_auc > best_auc:   # NaN-safe improvement check
        best_auc = video_auc
        state["best_metric"] = best_auc
        torch.save(state, RUN_DIR / "best.pt")
        bad_epochs = 0
        print(f"  new best video AUC {best_auc:.4f} -> saved best.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping: no val video-AUC improvement in {patience} epochs.")
            break

    elapsed_h = (time.time() - TRAIN_START) / 3600
    # Average over epochs run THIS SESSION, not the absolute epoch number:
    # on a resumed run `epoch` starts high (e.g. 9) while only 1-2 epochs have
    # actually elapsed, so elapsed/(epoch+1) badly UNDER-estimates the epoch
    # cost and the guard lets one epoch too many start -- straight into the
    # 12h kill, which publishes nothing.
    _epochs_this_run = max(1, epoch - start_epoch + 1)
    epoch_est_h = elapsed_h / _epochs_this_run     # measured average, not a guess
    projected_h = elapsed_h + epoch_est_h + EVAL_RESERVE_HOURS
    if projected_h > TIME_BUDGET_HOURS:
        print(f"TIME BUDGET: stopping after epoch {epoch:02d}. elapsed {elapsed_h:.1f}h "
              f"+ next epoch ~{epoch_est_h:.1f}h + eval {EVAL_RESERVE_HOURS}h "
              f"= {projected_h:.1f}h > {TIME_BUDGET_HOURS}h budget.")
        print("  -> another epoch would not leave room to finish the eval, and a "
              "killed commit publishes NOTHING. Resume from last.pt next run.")
        break

if not (RUN_DIR / "best.pt").exists():
    # No epoch this run beat the restored best_auc. The weights that actually
    # achieved best_auc live in the PREVIOUS run's best.pt (attached as input)
    # -- carry THAT, never the final-epoch weights (which are known-worse).
    # => the shipped best.pt is monotonically non-decreasing in val AUC.
    _prev_best = [p for p in sorted(_glob.glob("/kaggle/input/**/best.pt", recursive=True))
                  if "smoke" not in p]
    if _prev_best:
        shutil.copy(_prev_best[-1], RUN_DIR / "best.pt")
        print(f"no new best this run -> carried previous best.pt (val AUC {best_auc:.4f}) forward")
    else:
        carry = state if "state" in globals() else _resume
        assert carry is not None, "nothing to evaluate: no checkpoint produced or resumed"
        torch.save(carry, RUN_DIR / "best.pt")
        print("no new best this run -> carried forward final checkpoint as best.pt")

print(f"\nDone. Best val video AUC: {best_auc:.4f} | checkpoints in {RUN_DIR}")

## Stage 3.5: Calibrate the verdict thresholds (on VALIDATION only)

The model RANKS real-vs-AI well (high AUC) but the default 0.5 cutoff sits in
the wrong place -> reals get over-flagged as fake. Calibration fixes the
OPERATING POINT without touching the weights:

- **decision threshold t\*** = the cutoff maximizing balanced accuracy on the
  **validation** set (never test -- tuning on test would leak it).
- **verdict bands** = the widest zones around t\* whose val purity is >=90%:
  scores below `real_below` were >=90% real on val, above `fake_above` >=90%
  AI; in between the honest answer is "uncertain".

The calibrated cutoffs are WRITTEN INTO best.pt's config, so inference and the
frontend pick them up automatically -- the checkpoint stays self-describing.

In [ ]:
# ================= CALIBRATE verdict thresholds (VAL ONLY) ===================
ckpt = torch.load(RUN_DIR / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
val_windows, _ = collect_window_probs(model, val_loader, DEVICE)
vv = aggregate_videos(val_windows)
y = vv["label"].to_numpy().astype(int)
p = vv["mean_prob"].to_numpy()
assert len(np.unique(y)) == 2, "validation set must contain both classes to calibrate"

def bal_acc(t):
    pred = (p >= t).astype(int)
    tpr = ((pred == 1) & (y == 1)).sum() / max((y == 1).sum(), 1)
    tnr = ((pred == 0) & (y == 0)).sum() / max((y == 0).sum(), 1)
    return (tpr + tnr) / 2

ths = np.linspace(0.05, 0.95, 181)
scores = np.array([bal_acc(t) for t in ths])
t_star = float(ths[int(scores.argmax())])

# purity bands around t*: widest real zone / fake zone with >=90% val purity
PURITY, MIN_N = 0.90, 20
real_below = t_star
for a in np.arange(t_star, 0.02, -0.005):
    m = p < a
    if m.sum() >= MIN_N and (y[m] == 0).mean() >= PURITY:
        real_below = float(a)
        break
fake_above = t_star
for b in np.arange(t_star, 0.98, 0.005):
    m = p > b
    if m.sum() >= MIN_N and (y[m] == 1).mean() >= PURITY:
        fake_above = float(b)
        break

pred = (p >= t_star).astype(int)
tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
frac_unc = float(((p >= real_below) & (p <= fake_above)).mean())
print(f"decision threshold t* = {t_star:.3f}  (val balanced acc {scores.max():.4f})")
print(f"val @ t*: acc={(tp+tn)/len(y):.4f}  precision={tp/max(tp+fp,1):.4f}  "
      f"recall={tp/max(tp+fn,1):.4f}  FPR={fp/max(fp+tn,1):.4f}")
print(f"verdict bands (>= {PURITY:.0%} val purity): real < {real_below*100:.1f}  |  "
      f"uncertain  |  fake > {fake_above*100:.1f}   ({frac_unc:.1%} of val lands uncertain)")

# write the calibration INTO the checkpoint -- inference reads these fields
ckpt["config"]["inference"]["verdict_real_below"] = round(real_below * 100, 1)
ckpt["config"]["inference"]["verdict_fake_above"] = round(fake_above * 100, 1)
ckpt["config"]["inference"]["decision_threshold"] = round(t_star, 4)
ckpt["config"]["inference"]["calibration"] = {
    "method": "balanced-accuracy t* + 90%-purity bands", "split": "val",
    "n_val_videos": int(len(y)), "val_balanced_acc": round(float(scores.max()), 4),
}
torch.save(ckpt, RUN_DIR / "best.pt")
print("calibration written into best.pt (self-describing checkpoint)")

## Stage 4: Held-out test evaluation

Ports `evaluate.py`. Runs the **best** checkpoint on the **test** split (never
seen during training or early-stopping decisions), aggregated to the video
level, with metrics broken out **per source** -- so you can see real-vs-AI
performance cleanly.

**Sliced eval (shortcut detector).** It then breaks AUC out **by orientation**
(portrait / landscape / square) and **by resolution** (short side: <=360p ...
>1080p), using the original pre-resize dimensions recorded at extraction. Read
the two flags:

- **`[SHORTCUT RISK]`** -- a slice is nearly one class (e.g. portrait is 92%
  fake). That means the factor is *correlated with the label in your data*, so
  the model can "detect AI" just by reading orientation/resolution. Fix it in
  the **data** (add the missing class for that slice), not in the model.
- **`[WEAK SLICE]`** -- AUC is much lower on some slice (e.g. <=360p). The model
  learned a cue that doesn't hold there; add examples / lean on augmentation.

A high overall AUC with a clean slice table is what you actually want -- it's
evidence the model generalizes across orientations and resolutions instead of
exploiting one. (Slices show `unknown` for any frames extracted before dims
were recorded -- re-extract to populate them.)

In [ ]:
# ============================== EVALUATE ON TEST SPLIT =========================
import json as _json

ckpt = torch.load(RUN_DIR / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])

test_df = manifest_df[manifest_df["split"] == "test"]
assert not test_df.empty, "no rows in the test split"

test_ds = WindowDataset(test_df, build_window_transforms(CFG, train=False), CFG["paths"]["data_root"], with_meta=True)
test_loader = DataLoader(test_ds, batch_size=CFG["train"]["batch_videos"], shuffle=False,
                         num_workers=CFG["train"]["num_workers"], pin_memory=DEVICE.type == "cuda")

window_df, _ = collect_window_probs(model, test_loader, DEVICE)
video_df = aggregate_videos(window_df)
video_df = add_slice_columns(video_df, test_df)   # join orig_w/orig_h -> orientation, resolution

results = {
    "split": "test",
    "overall": compute_metrics(video_df),
    "per_source": per_source_metrics(video_df),
    "by_orientation": sliced_metrics(video_df, "orientation"),
    "by_resolution": sliced_metrics(video_df, "resolution"),
    "by_generator": per_generator_metrics(video_df),
    # branch contribution: same run, same test videos, three scoring heads
    "branch_auc": {
        "fused": safe_auc(video_df["label"], video_df["mean_prob"]),
        "spatial_only": safe_auc(video_df["label"], video_df["mean_prob_spatial"]),
        "temporal_only": safe_auc(video_df["label"], video_df["mean_prob_temporal"]),
    },
}
if "mean_prob_frequency" in video_df.columns:
    results["branch_auc"]["frequency_only"] = safe_auc(video_df["label"], video_df["mean_prob_frequency"])
if "mean_prob_motion" in video_df.columns:
    results["branch_auc"]["motion_only"] = safe_auc(video_df["label"], video_df["mean_prob_motion"])

print(f"=== TEST (video-level, {results['overall']['n_videos']} videos) ===")
print(_json.dumps(results["overall"], indent=2))

# Metrics at the CALIBRATED operating point (t* chosen on val -> no test
# leakage). This is the accuracy the deployed system actually delivers;
# the block above (0.5 cutoff) is kept for comparison.
t_star = float(ckpt["config"]["inference"].get("decision_threshold", 0.5))
results["overall_at_calibrated"] = compute_metrics(video_df, threshold=t_star)
oc = results["overall_at_calibrated"]
print(f"\n=== at CALIBRATED threshold t*={t_star:.3f} (chosen on VAL) ===")
print(f"  accuracy {oc['accuracy']:.4f} | precision {oc['precision']:.4f} | "
      f"recall {oc['recall']:.4f} | f1 {oc['f1']:.4f}")
print(f"  confusion {oc['confusion_matrix']}")

# THE thesis table: what each branch contributes, from ONE jointly-trained model.
ba = results["branch_auc"]
print("\n=== branch contribution (video AUC) ===")
print(f"  spatial branch alone : {ba['spatial_only']:.4f}   (single-frame texture artifacts)")
print(f"  temporal branch alone: {ba['temporal_only']:.4f}   (ConvLSTM motion dynamics)")
if "frequency_only" in ba:
    print(f"  frequency branch alone: {ba['frequency_only']:.4f}  (FFT spectral fingerprints)")
if "motion_only" in ba:
    print(f"  motion branch alone  : {ba['motion_only']:.4f}   (temporal-residual incoherence)")
print(f"  FUSED (the model)    : {ba['fused']:.4f}")
_best_branch = max(v for k, v in ba.items() if k != "fused")
print(f"  fused vs best branch : {ba['fused'] - _best_branch:+.4f}  "
      f"({'fusion HELPS' if ba['fused'] >= _best_branch else 'fusion below best branch'})")
# Does the model actually USE each branch? Print the learned gate weights.
if getattr(model, "use_logit_fusion", False):
    _members = (["mlp_fusion", "spatial", "temporal"]
                + (["frequency"] if model.use_frequency else [])
                + (["motion"] if model.use_motion else []))
    _w = torch.softmax(model.fusion_gate.detach().cpu(), dim=0).tolist()
    print("  learned fusion weights: " + ", ".join(f"{n}={w:.2f}" for n, w in zip(_members, _w)))
    print("    (a branch the model trusts gets meaningful weight; ~0 => it is ignored / dead weight)")

for source, metrics in results["per_source"].items():
    print(f"--- real vs {source} ---")
    print(_json.dumps(metrics, indent=2))

# Shortcut instrumentation: AUC broken out by orientation and by resolution.
# Read the flags -- [SHORTCUT RISK] = a slice is one-class (data confound);
# [WEAK SLICE] = model is much weaker on some slice values.
print()
print(format_slice_report("orientation", results["by_orientation"]))
print(format_slice_report("resolution", results["by_resolution"]))
if results["by_generator"]:
    print()
    print(format_generator_report(results["by_generator"]))

# ROUND-2 SUCCESS METRIC: are wild reals (ugc/vision/pexels) still being
# flagged as AI at the deployed threshold, or do they now match stock reals?
results["by_real_source_fpr"] = per_real_source_fpr(video_df, t_star)
if results["by_real_source_fpr"]:
    print()
    print(format_real_source_report(results["by_real_source_fpr"], t_star))

# Portrait robustness probe: re-score the test set forced to 9:16 portrait.
# Small AUC drop = squash-invariant = portrait uploads handled despite
# all-landscape training. Big drop = portrait is OOD -> get portrait data.
probe_tf = A.ReplayCompose(build_portrait_probe_transforms(CFG).transforms)
probe_ds = WindowDataset(test_df, probe_tf, CFG["paths"]["data_root"], with_meta=True)
probe_loader = DataLoader(probe_ds, batch_size=CFG["train"]["batch_videos"], shuffle=False,
                          num_workers=CFG["train"]["num_workers"], pin_memory=DEVICE.type == "cuda")
probe_frame_df, _ = collect_window_probs(model, probe_loader, DEVICE)
probe_video = aggregate_videos(probe_frame_df)
probe_auc = safe_auc(probe_video["label"], probe_video["mean_prob"])
base_auc = results["overall"]["auc"]
drop = (base_auc - probe_auc) if probe_auc == probe_auc else None
results["portrait_probe"] = {"auc": probe_auc, "base_auc": base_auc, "auc_drop": drop}
if drop is not None:
    verdict = ("robust" if drop < 0.03 else "acceptable" if drop < 0.07 else "WEAK -> get portrait data")
    print(f"\n=== portrait robustness probe (landscape test forced to 9:16) ===")
    print(f"  normal AUC {base_auc:.4f} -> portrait-squash AUC {probe_auc:.4f} (drop {drop:.4f}) -> {verdict}")

with open(RUN_DIR / "metrics_test.json", "w") as f:
    _json.dump(results, f, indent=2)
print(f"\nSaved -> {RUN_DIR / 'metrics_test.json'}")
print(f"\nAll run artifacts (checkpoints, curves, metrics) are under {RUN_DIR}")
print("They will appear in this notebook's Output tab after Save & Run All (Commit).")

## Next steps
1. **If this was the smoke test:** check `curves.png` and the printed AUCs
   above -- with only ~120 videos and 3 epochs, don't expect a high AUC; the
   goal is just confirming nothing crashed and the numbers look sane (not
   NaN, not stuck at exactly 0.5 which would suggest a labeling bug). If
   clean, set `SMOKE_TEST = False` and **Save Version -> Save & Run All
   (Commit)** for the real run.
2. **If this was the full run:** check `best.pt`'s test AUC in the printed
   `per_source` breakdown. A model that's strong on `ai_generated` but weak
   elsewhere, or vice versa, tells you where to focus next (more data,
   more variety in that source).
3. **Using the trained model:** download `best.pt` from the Output tab into
   `training/models/`. The checkpoint stores `arch: hybrid` in its config;
   `training/src/inference.py` detects it, reads contiguous windows from the
   uploaded video, and produces the frontend JSON (fused verdict + GradCAM
   boxes from the spatial branch).
4. **Adding data later (e.g. the portrait batch):** extract ONLY the new
   videos, attach both extraction outputs here, and retrain FROM SCRATCH on
   the combined pool (never fine-tune this checkpoint on just the new slice
   -- that causes catastrophic forgetting of what it already learned).